# General information
<span style="color: green"> **Please use the BIDS structure**</span> so that pipeline can run smoothly. Place ipynb-notebooks on the top level of the project folder (same level as subject folders).


**Gereral processing steps:**  

0. [Import libraries](#0-import-libraries)  
1. [Set channel types and montage](#1-channel-types-montage)  
2. [Filter](#2-filter)  
3. [Restoring Reference channel](#3-restoring-reference-channel-and-average-reference)  
4. [Epoching](#4-epoching) 
5. [GEDAI](#5-gedai)  
6. [Adaptive Mixture Independent Component Analysis](#6-adaptive-mixture-independent-component-analysis)  

**Outline of this codebook**:   
1. [Setup Preprocessing (Step 1.-4.)](#1-set-up-preprocessing)
2. [Run Preprocessing (Step 1.-4.)](#2-run-preprocessing)
5. [GEDAI](#5-gedai)  
6. [Adaptive Mixture Independent Component Analysis](#6-adaptive-mixture-independent-component-analysis)  

::: {#eeg-process}

![](Fig2-EEGProcess.svg){width=80%}

EEG processing overview

:::

# 0. Import libraries

The analysis pipeline is based on the following libraries. In case of an error in the execution of this cell, probably one or more of the libraries is not installed. In this case, start a terminal in **ANACONDA.NAVIGATOR** *Environments>Terminal* and install the library in question using the command <span style='color: red'>*pip install [library name]*</span>.

In [5]:
import warnings                 # switch of pandas and mne warnings
warnings.simplefilter(action='ignore')

import requests
import os
import os.path as op
import time
from pathlib import Path

import mne

import gc
import numpy as np
import pandas as pd
import ipyfilechooser
import ipywidgets as widgets
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import pyprep
from pyprep.prep_pipeline import PrepPipeline
from datetime import datetime, timedelta
from datetime import datetime, timezone
#from playsound import playsound
from openpyxl import load_workbook
from mne.export import export_raw

from collections import defaultdict
from mne.datasets import fetch_fsaverage
from mne.evoked import combine_evoked
from mne.forward import make_forward_dipole
from mne.simulation import simulate_evoked
from mne.viz import circular_layout
from mne_connectivity import spectral_connectivity_epochs
from mne_connectivity.viz import plot_connectivity_circle
from mne_icalabel import label_components
from mne_icalabel.gui import label_ica_components
#from asrpy import ASR
from meegkit.asr import ASR
from gedai import Gedai, AdaptiveMultibandGedai
from gedai.viz import plot_mne_style_overlay_interactive

from nilearn.image import index_img
from nilearn.plotting import plot_stat_map
from mne import read_evokeds
from mne.datasets import sample
from mne.minimum_norm import  make_inverse_operator, apply_inverse, read_inverse_operator, apply_inverse_epochs

from pyamica import AMICA, AmicaICA
import torch

%matplotlib qt

mne.set_log_level("ERROR")  # only show errors (no warnings or information messages)

In [6]:
wd = r"D:\BIDS_CoMoCut\BIDS\CoMoCut_Proof-of-concept"
os.chdir(wd)
print(os.getcwd())

D:\BIDS_CoMoCut\BIDS\CoMoCut_Proof-of-concept


# 1. Set up Preprocessing
- Loading data
- Set montage
- Filter
- Restore ref channel

In [3]:
# -------------------------------------------------------------------------
# FUNCTION: add_ref
# Adds FCz back as a zero channel (online recording reference) and
# re-references to average. The montage is re-applied after adding FCz
# since adding a new channel resets electrode locations.
#
# Parameters
# ----------
# raw : mne.io.Raw — filtered raw EEG data without FCz
#
# Returns
# -------
# raw : mne.io.Raw — re-referenced raw EEG data with FCz restored
# -------------------------------------------------------------------------
def add_ref(raw):
    if "FCz" not in raw.ch_names:
        raw = mne.add_reference_channels(raw, ref_channels=["FCz"])
    else:
        print("FCz already present, skipping")

    raw.set_montage(mne.channels.make_standard_montage("easycap-M1"))
    raw.set_eeg_reference("average", projection=False)
    return raw

# 2. Run preprocessing

In [ ]:
wd = Path.cwd()
CONCAT_RAW_PATH = wd / "derivatives" / "02_concat-raw"
PREPROC_PATH    = wd / "derivatives" / "02x_raw_preproc"

L_FREQ = 1     # high-pass cutoff (Hz)
H_FREQ = 131   # low-pass cutoff (Hz) — 131 for high-pass only (includes 131 Hz online filter)

raw_files = sorted(CONCAT_RAW_PATH.rglob("*desc-concat_eeg.fif"))
print(f"Found {len(raw_files)} concatenated raw files")

for fif_file in raw_files:
    ses_name = fif_file.parent.parent.name
    sub_name = fif_file.parent.parent.parent.name

    print(f"\nProcessing : {sub_name} / {ses_name}")

    try:
        # --- LOAD ---
        raw = mne.io.read_raw_fif(fif_file, preload=True)

        # --- CHANNEL TYPES AND MONTAGE ---
        raw.set_channel_types({
            "HEOG"   : "eog",
            "VEOG"   : "eog",
            "NeckEMG": "emg",
            "x_dir"  : "misc",
            "y_dir"  : "misc",
            "z_dir"  : "misc",
        })
        raw.set_montage(mne.channels.make_standard_montage("easycap-M1"))

        # --- FILTER ---
        raw.filter(l_freq=L_FREQ, h_freq=H_FREQ)

        # --- RESTORE FCz + AVERAGE REFERENCE ---
        raw = add_ref(raw)

        # --- SAVE ---
        output_dir = PREPROC_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = f"{sub_name}_{ses_name}_task-sidecut_desc-preproc_eeg.fif"
        output_path = output_dir / output_name
        raw.save(output_path, overwrite=True)
        print(f"  Saved : {output_name}")

    except Exception as e:
        print(f"  Error : {sub_name} / {ses_name} : {e}")

    finally:
        if "raw" in locals():
            del raw
        gc.collect()

# 3. GEDAI

Use the Generalized Eigenvalue De-Artifacting Instrument (GEDAI) from [@rosReturnGEDAIUnsupervised2025].

Apply GEDAI directly on the continuous preprocessed raw data (not yet epoched). The current GEDAI implementation determines its own analysis window internally — a fixed window for the broadband pass, and per-frequency-band windows sized automatically from the sampling rate for the spectral pass — so no manual epoching is required beforehand. First use the general (broadband) GEDAI. Then, use the adaptive spectral version for more aggressive and accurate cleaning.

Link:
- *GEDAI*: [https://github.com/neurotuning/gedai](https://github.com/neurotuning/gedai)

## 3.1. Broadband GEDAI

Fits and applies GEDAI on the continuous raw recording, using fixed-length internal windows (`GEDAI_DURATION` / `GEDAI_OVERLAP` below).

In [ ]:
wd           = Path.cwd()
PREPROC_PATH = wd / "derivatives" / "02x_raw_preproc"
GEDAI_PATH   = wd / "derivatives" / "03a_gedai"

# -------------------------------------------------------------------------
# CONFIGURATION
# GEDAI expects EEG-only channels — its leadfield reference covariance is
# built from 10-5 system electrode names, so non-EEG channels (EOG, EMG,
# misc) must be excluded before fitting/transforming, then reattached
# afterward.
# Applied directly on the continuous preprocessed raw data: GEDAI internally
# cuts it into fixed-length windows (duration/overlap below) to fit and
# clean — no prior epoching needed.
# Output: derivatives/03a_gedai/sub-XX/ses-XX/eeg/
# -------------------------------------------------------------------------
GEDAI_DURATION         = 1.0
GEDAI_OVERLAP          = 0.5
GEDAI_REJECT_BY_ANNOT  = False
GEDAI_REFERENCE_COV    = "leadfield"
GEDAI_SENSAI_METHOD    = "gridsearch"
GEDAI_NOISE_MULTIPLIER = "auto"
# -------------------------------------------------------------------------
raw_files = sorted(PREPROC_PATH.rglob("*desc-preproc_eeg.fif"))
print(f"Found {len(raw_files)} preprocessed raw files")

for fif_path in raw_files:
    ses_name = fif_path.parent.parent.name
    sub_name = fif_path.parent.parent.parent.name

    print(f"\nGEDAI : {sub_name} / {ses_name}")

    try:
        raw = mne.io.read_raw_fif(fif_path, preload=True)

        raw_eeg = raw.copy().pick("eeg")
        non_eeg_picks = mne.pick_types(raw.info, eeg=False, eog=True, emg=True, misc=True)
        raw_non_eeg = raw.copy().pick(non_eeg_picks) if len(non_eeg_picks) > 0 else None

        gedai = Gedai()
        gedai.fit_raw(
            raw_eeg,
            duration             = GEDAI_DURATION,
            overlap              = GEDAI_OVERLAP,
            reject_by_annotation = GEDAI_REJECT_BY_ANNOT,
            reference_cov        = GEDAI_REFERENCE_COV,
            sensai_method        = GEDAI_SENSAI_METHOD,
            noise_multiplier     = GEDAI_NOISE_MULTIPLIER,
            verbose              = False,
        )

        fig = gedai.plot_fit()
        plt.show()

        raw_trans_eeg = gedai.transform_raw(
            raw_eeg,
            overlap = GEDAI_OVERLAP,
            verbose = False,
        )

        # Reattach non-EEG channels
        if raw_non_eeg is not None:
            with raw_non_eeg.info._unlock():
                raw_non_eeg.info["custom_ref_applied"] = raw_trans_eeg.info["custom_ref_applied"]
            raw_trans = raw_trans_eeg.copy().add_channels([raw_non_eeg])
        else:
            raw_trans = raw_trans_eeg

        output_dir = GEDAI_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = fif_path.name.replace("_desc-preproc_eeg.fif", "_desc-gedai_eeg.fif")
        output_path = output_dir / output_name

        raw_trans.save(output_path, overwrite=True)
        print(f"  Saved : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} : {ex}")

    finally:
        for var in ["raw", "raw_eeg", "raw_non_eeg", "gedai", "raw_trans_eeg", "raw_trans"]:
            if var in locals():
                del locals()[var]
        gc.collect()

Found 21 preprocessed raw files

GEDAI : sub-01 / ses-01
  Saved : sub-01_ses-01_task-sidecut_desc-gedai_eeg.fif

GEDAI : sub-02 / ses-01
  Saved : sub-02_ses-01_task-sidecut_desc-gedai_eeg.fif

GEDAI : sub-03 / ses-01


In [ ]:
# -------------------------------------------------------------------------
# INSPECT GEDAI RESULT
# Compares the original preprocessed raw against the saved GEDAI output
# for one subject/session. Uses already-processed files — no refitting
# needed.
# -------------------------------------------------------------------------
wd           = Path.cwd()
PREPROC_PATH = wd / "derivatives" / "02x_raw_preproc"
GEDAI_PATH   = wd / "derivatives" / "03a_gedai"

INSPECT_SUB = "sub-01"
INSPECT_SES = "ses-01"

orig_matches  = list((PREPROC_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob("*desc-preproc_eeg.fif"))
gedai_matches = list((GEDAI_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob("*desc-gedai_eeg.fif"))

if not orig_matches or not gedai_matches:
    print(f"Missing file(s) for {INSPECT_SUB} / {INSPECT_SES} — "
          f"original: {len(orig_matches)}, GEDAI: {len(gedai_matches)}")
else:
    raw_orig  = mne.io.read_raw_fif(orig_matches[0], preload=True)
    raw_gedai = mne.io.read_raw_fif(gedai_matches[0], preload=True)

## 3.2. Spectral GEDAI

Adaptive multiband GEDAI: the wavelet decomposition level and each sub-band's analysis window length are determined automatically from the data's sampling rate (`wavelet_level="auto"`, `cycles_per_wavelet`), rather than fixed by hand.

In [ ]:
wd                  = Path.cwd()
GEDAI_PATH          = wd / "derivatives" / "03a_gedai"
GEDAI_SPECTRAL_PATH = wd / "derivatives" / "03b_gedai_gedaiSpec"

# -------------------------------------------------------------------------
# CONFIGURATION
# Spectral GEDAI: same algorithm as broadband GEDAI, but with a wavelet
# decomposition splitting the data into multiple frequency sub-bands.
# Each sub-band gets its own SENSAI threshold and its own analysis window
# length, automatically sized to fit enough cycles of that band's lowest
# frequency (cycles_per_wavelet below) — no manual per-band tuning needed.
# wavelet_level="auto" is likewise computed from the sampling rate.
# broadband_pass is disabled here since the broadband pass already ran
# in 3.1 above.
# Output: derivatives/03b_gedai_gedaiSpec/sub-XX/ses-XX/eeg/
# -------------------------------------------------------------------------
GEDAI_WAVELET_TYPE       = "haar"  # wavelet basis
GEDAI_WAVELET_LEVEL      = "auto"  # decomposition level, or an int to override
GEDAI_CYCLES_PER_WAVELET = 10      # cycles used to size each band's window
GEDAI_REFERENCE_COV      = "leadfield"
GEDAI_SENSAI_METHOD      = "gridsearch"
GEDAI_NOISE_MULTIPLIER   = "auto"
# -------------------------------------------------------------------------
gedai_files = sorted(GEDAI_PATH.rglob("*desc-gedai_eeg.fif"))
print(f"Found {len(gedai_files)} GEDAI-cleaned raw files to process")

for fif_path in gedai_files:
    ses_name = fif_path.parent.parent.name
    sub_name = fif_path.parent.parent.parent.name

    print(f"\nSpectral GEDAI : {sub_name} / {ses_name}")

    try:
        raw = mne.io.read_raw_fif(fif_path, preload=True)

        raw_eeg = raw.copy().pick("eeg")
        non_eeg_picks = mne.pick_types(raw.info, eeg=False, eog=True, emg=True, misc=True)
        raw_non_eeg = raw.copy().pick(non_eeg_picks) if len(non_eeg_picks) > 0 else None

        gedai_spectral = AdaptiveMultibandGedai(
            wavelet_type       = GEDAI_WAVELET_TYPE,
            wavelet_level      = GEDAI_WAVELET_LEVEL,
            cycles_per_wavelet = GEDAI_CYCLES_PER_WAVELET,
            broadband_pass     = False,
        )
        gedai_spectral.fit_raw(
            raw_eeg,
            reference_cov    = GEDAI_REFERENCE_COV,
            sensai_method    = GEDAI_SENSAI_METHOD,
            noise_multiplier = GEDAI_NOISE_MULTIPLIER,
            verbose          = False,
        )

        fig = gedai_spectral.plot_fit()
        plt.show()

        raw_trans_eeg = gedai_spectral.transform_raw(
            raw_eeg,
            verbose = False,
        )

        # Reattach non-EEG channels
        if raw_non_eeg is not None:
            with raw_non_eeg.info._unlock():
                raw_non_eeg.info["custom_ref_applied"] = raw_trans_eeg.info["custom_ref_applied"]
            raw_trans = raw_trans_eeg.copy().add_channels([raw_non_eeg])
        else:
            raw_trans = raw_trans_eeg

        output_dir = GEDAI_SPECTRAL_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = fif_path.name.replace("_desc-gedai_eeg.fif", "_desc-gedaiSpectral_eeg.fif")
        output_path = output_dir / output_name

        raw_trans.save(output_path, overwrite=True)
        print(f"  Saved : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} : {ex}")

    finally:
        for var in ["raw", "raw_eeg", "raw_non_eeg", "gedai_spectral", "raw_trans_eeg", "raw_trans"]:
            if var in locals():
                del locals()[var]
        gc.collect()

In [ ]:
# -------------------------------------------------------------------------
# INSPECT SPECTRAL GEDAI RESULT
# Compares the original preprocessed raw against the saved GEDAI output
# for one subject/session. Uses already-processed files — no refitting
# needed.
# -------------------------------------------------------------------------
wd              = Path.cwd()
PREPROC_PATH    = wd / "derivatives" / "02x_raw_preproc"
GEDAI_PATH      = wd / "derivatives" / "03a_gedai"
SPEC_GEDAI_PATH = wd / "derivatives" / "03b_gedai_gedaiSpec"

INSPECT_SUB = "sub-01"
INSPECT_SES = "ses-01"

orig_matches       = list((PREPROC_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob("*desc-preproc_eeg.fif"))
gedai_matches      = list((GEDAI_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob("*desc-gedai_eeg.fif"))
spec_gedai_matches = list((SPEC_GEDAI_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob("*desc-gedaiSpectral_eeg.fif"))

if not orig_matches or not gedai_matches or not spec_gedai_matches:
    print(f"Missing file(s) for {INSPECT_SUB} / {INSPECT_SES} — "
          f"original: {len(orig_matches)}, GEDAI: {len(gedai_matches)}, Spectral GEDAI: {len(spec_gedai_matches)}")
else:
    raw_orig      = mne.io.read_raw_fif(orig_matches[0], preload=True)
    raw_gedai     = mne.io.read_raw_fif(gedai_matches[0], preload=True)
    raw_specgedai = mne.io.read_raw_fif(spec_gedai_matches[0], preload=True)

# 4. Epoching

In [ ]:
# -------------------------------------------------------------------------
# EPOCH CONFIGURATION
# Each entry corresponds to one epoch type (RS_left, RS_right, IC_left, IC_right).
# Originally: RS = (-4.5, 0.5); IC = (-5.0, 0)
# New version (check after IC): RS = (-4.5, 1.5); IC = (-5.0, 1.0)
# -------------------------------------------------------------------------
scal = {"eeg": 1e-4, "eog": 1e-4, "emg": 1e2, "misc": 1e3}

epoch_names  = ["RS_left", "RS_right", "IC_left", "IC_right"]
epoch_range  = [(-4.5, 1.5), (-4.5, 1.5), (-5.0, 1.0), (-5.0, 1.0)]
epoch_base   = [(-4.5, -2.5), (-4.5, -2.5), (-5.0, -3.0), (-5.0, -3.0)]
epoch_reject = [(-1.5, -1.0), (-1.5, -1.0), (-2.0, -1.5), (-2.0, -1.5)]

In [ ]:
# -------------------------------------------------------------------------
# FUNCTION: make_epochs
# Creates epochs for a single annotation type from a Raw object.
#
# Parameters
# ----------
# raw  : mne.io.Raw — preprocessed raw data
# i    : int — index into epoch_names/range/base/reject lists
#
# Returns
# -------
# epochs : mne.Epochs
# -------------------------------------------------------------------------
def make_epochs(raw, i):
    evt, _ = mne.events_from_annotations(raw, event_id={epoch_names[i]: 1001})

    epochs = mne.Epochs(
        raw,
        events    = evt,
        event_id  = {epoch_names[i]: 1001},
        tmin      = epoch_range[i][0],
        tmax      = epoch_range[i][1],
        baseline  = epoch_base[i],
        #reject    = {"eeg": 500e-6},
        flat      = {"eeg": 1e-10},
        detrend   = 1,
        preload   = True,
    )
    return epochs

In [ ]:
# -------------------------------------------------------------------------
# FUNCTION: filter_by_atr
# Rejects epochs with ATR exceeding ATR_MAX. If ATR_MAX is None, the
# threshold is computed as mean + 3 SD of the ATR distribution per subject.
# Metadata is updated to reflect only the retained epochs.
#
# Parameters
# ----------
# epochs  : mne.Epochs — epoch object with ATR column in metadata
# atr_max : float | None — upper ATR bound in ms (None = mean + 3 SD)
#
# Returns
# -------
# filtered_epochs : mne.Epochs — epochs with ATR <= atr_max
# -------------------------------------------------------------------------
def filter_by_atr(epochs, atr_max):
    if epochs.metadata is None or "ATR" not in epochs.metadata.columns:
        raise ValueError("No ATR metadata found in epochs")

    atr_values = epochs.metadata["ATR"].values

    # Resolve None threshold from data distribution (mean + 3 SD)
    hi = (np.mean(atr_values) + 3 * np.std(atr_values)) if atr_max is None else atr_max

    keep_idx   = np.where(atr_values <= hi)[0]
    n_rejected = len(epochs) - len(keep_idx)

    # Handle None filename for in-memory epochs (not yet saved to disk)
    fname = Path(epochs.filename).name if epochs.filename is not None else "in-memory"

    print(f"  {fname} : "
          f"ATR <= {hi:.0f} ms → "
          f"{len(keep_idx)} kept / {n_rejected} rejected")

    filtered          = epochs[keep_idx]
    filtered.metadata = filtered.metadata.reset_index(drop=True)

    return filtered

In [ ]:
# -------------------------------------------------------------------------
# LOAD, EPOCH, ADD ATR, AND SAVE — ONE FILE AT A TIME
# Combines loading and epoching so only one subject's raw recording exists
# in memory at any point, rather than holding all subjects' GEDAI-cleaned
# data simultaneously.
# Discovers all .fif files under 03b_gedai_gedaiSpec/sub-XX/ses-XX/eeg/,
# creates epochs for all four annotation types (RS/IC × left/right),
# computes ATR (anticipatory time to response = IC onset − RS onset) as
# epoch metadata, rejects trials with excessively long ATR, and saves.
# Output: 04_gedai_gedaiSpec_epochs/sub-XX/ses-XX/eeg/
# -------------------------------------------------------------------------
wd                  = Path.cwd()
GEDAI_SPECTRAL_PATH = wd / "derivatives" / "03b_gedai_gedaiSpec"
EPOCHS_PATH         = wd / "derivatives" / "04_gedai_gedaiSpec_epochs"
ATR_MAX             = 1500  # ms — trials with ATR > this value are rejected

gedai_files = sorted(GEDAI_SPECTRAL_PATH.rglob("*desc-gedaiSpectral_eeg.fif"))
print(f"Found {len(gedai_files)} GEDAI-cleaned raw files")

for fif_file in gedai_files:
    ses_name = fif_file.parent.parent.name
    sub_name = fif_file.parent.parent.parent.name

    print(f"\nProcessing : {sub_name} / {ses_name}")

    try:
        # --- LOAD ---
        raw = mne.io.read_raw_fif(fif_file, preload=True)

        # --- EPOCHING (all four types, needed together for ATR below) ---
        epochs_by_label = {}
        for i, label in enumerate(["RS_left", "RS_right", "IC_left", "IC_right"]):
            epochs = make_epochs(raw, i)
            n_total = len(epochs.drop_log)  # before rejection
            n_kept  = len(epochs)           # after rejection
            print(f"  {label:<12} : {n_kept} kept / {n_total - n_kept} rejected / {n_total} total")
            epochs_by_label[label] = epochs

        # Raw is no longer needed once all four epoch types are extracted
        del raw
        gc.collect()

        # --- ATR METADATA, REJECT LONG ATR TRIALS, SAVE ---
        output_dir = EPOCHS_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        for label in ["RS_left", "RS_right", "IC_left", "IC_right"]:
            epochs = epochs_by_label[label]
            side = label.split("_")[1]

            # ATR requires both the RS and IC epochs for the same side
            rs_epochs = epochs_by_label[f"RS_{side}"]
            ic_epochs = epochs_by_label[f"IC_{side}"]

            rs_onsets = rs_epochs.events[:, 0] / rs_epochs.info["sfreq"]
            ic_onsets = ic_epochs.events[:, 0] / ic_epochs.info["sfreq"]

            if len(rs_onsets) == len(ic_onsets):
                atr_ms = (ic_onsets - rs_onsets) * 1000
            else:
                print(f"  Warning : ATR length mismatch for {label} "
                      f"({len(rs_onsets)} RS vs {len(ic_onsets)} IC) — filling with NaN")
                atr_ms = np.full(len(epochs), np.nan)

            meta = pd.DataFrame({
                "trial": np.arange(1, len(epochs) + 1),
                "ATR"  : atr_ms,
                "side" : side,
            })
            epochs.metadata = meta

            # Reject trials with ATR above ATR_MAX
            n_before = len(epochs)
            epochs   = filter_by_atr(epochs, atr_max=ATR_MAX)
            epochs.metadata = epochs.metadata.reset_index(drop=True)
            n_after  = len(epochs)

            print(f"  {label:<10} : {n_before - n_after} / {n_before} trials rejected "
                  f"(ATR > {ATR_MAX} ms), {n_after} retained")

            if n_after == 0:
                print(f"  Warning : no epochs remaining after ATR filter for {label} — skipping")
                continue

            # Report ATR summary statistics on retained trials only
            atr_retained = epochs.metadata["ATR"].values
            print(f"  ATR {side:<6} (retained) : "
                  f"n={n_after} | "
                  f"mean={np.nanmean(atr_retained):.1f} ± {np.nanstd(atr_retained):.1f} ms | "
                  f"range=[{np.nanmin(atr_retained):.1f}, {np.nanmax(atr_retained):.1f}] ms")

            # Save
            output_name = f"{sub_name}_{ses_name}_task-sidecut_{label}-epo.fif"
            output_path = output_dir / output_name
            epochs.save(output_path, overwrite=True)
            print(f"  Saved : {output_name}")

        del epochs_by_label
        gc.collect()

    except Exception as e:
        print(f"  Error : {sub_name} / {ses_name} : {e}")
        for var in ["raw", "epochs_by_label"]:
            if var in locals():
                del locals()[var]
        gc.collect()

# 5. Adaptive Mixture Independent Component Analysis

Run the python adaptation of the adaptive mixture ICA (AMICA) [@palmerAMICAAdaptiveMixture2011].  

Link:  
[https://github.com/DerAndereJohannes/pyamica](https://github.com/DerAndereJohannes/pyamica)

## 5.1. AMICA fitting

In [ ]:
# -------------------------------------------------------------------------
# AMICA ICA FITTING
# Fits AMICA on epochs for each subject/session, using GEDAI-cleaned data.
# Rank is computed per subject to set n_components automatically.
# Fitted AMICA objects are saved to 05_gedai_gedaiSpec_amica_proc/
# -------------------------------------------------------------------------
wd = Path.cwd()
EPOCHS_PATH = wd / "derivatives" / "04_gedai_gedaiSpec_epochs"
AMICA_PROC_PATH = wd / "derivatives" / "05_gedai_gedaiSpec_amica_proc"

AMICA_FIT_HPASS = 3.0 # Hz - high-pass filtering for AMICA fitting

# Cropping window (tmax) for the fitting copy, per epoch type
FIT_CROP_TMAX = {
    "IC": 0.0,
    "RS": 0.25,
}

# -------------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------------
AMICA_EPOCH_TYPES = None  # "IC", "RS", or None to process all epochs

if AMICA_EPOCH_TYPES == "IC":
    epoch_pattern = "*IC_*-epo.fif"
elif AMICA_EPOCH_TYPES == "RS":
    epoch_pattern = "*RS_*-epo.fif"
elif AMICA_EPOCH_TYPES is None:
    epoch_pattern = "*-epo.fif"
else:
    raise ValueError(f"Unknown AMICA_EPOCH_TYPES: '{AMICA_EPOCH_TYPES}'. Choose 'IC', 'RS', or None.")

epoch_files = sorted(EPOCHS_PATH.rglob(epoch_pattern))
print(f"Found {len(epoch_files)} epoch files to process (AMICA_EPOCH_TYPES='{AMICA_EPOCH_TYPES}')")

for fif_path in epoch_files:
    ses_name = fif_path.parent.parent.name
    sub_name = fif_path.parent.parent.parent.name

    for candidate in ["IC_left", "IC_right", "RS_left", "RS_right"]:
        if candidate in fif_path.name:
            label = candidate
            break
    else:
        print(f"  Could not determine label from filename, skipping : {fif_path.name}")
        continue

    print(f"\nFitting AMICA : {sub_name} / {ses_name} / {label}")

    try:
        e = mne.read_epochs(fif_path, preload=True)

        # Filter first (on the full-length epoch, for stable filter behavior),
        # then crop — so the filter edge doesn't land exactly at the new boundary
        epoch_type = label.split("_")[0]  # "IC" or "RS"
        crop_tmax  = FIT_CROP_TMAX[epoch_type]

        e_fit = e.copy().filter(l_freq=AMICA_FIT_HPASS, h_freq=None, picks="eeg")
        e_fit = e_fit.crop(tmax=crop_tmax)
        print(f"  Cropped fitting copy to tmax={crop_tmax}s ({epoch_type})")

        rank = mne.compute_rank(e_fit, tol=1e-6, tol_kind="relative")
        print(f"  Rank : {rank['eeg']}")

        amica = AmicaICA(
            n_models     = 1,
            n_components = rank["eeg"],
            do_reject    = True,
            reject_sigma = 3.0,
            num_reject   = 5,
            reject_int   = 1,
            device       = "cpu"
        )
        amica.fit(e_fit, picks="eeg")

        output_dir = AMICA_PROC_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = fif_path.name.replace("-epo.fif", "_amica")
        output_path = output_dir / output_name

        amica.save(str(output_path))
        print(f"  Saved AMICA : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} / {label} : {ex}")

    finally:
        for var in ["e", "e_fit", "amica"]:
            if var in locals():
                del locals()[var]
        gc.collect()

In [ ]:
# -------------------------------------------------------------------------
# OPTIONAL: INSPECT AMICA COMPONENTS
# Configure and run this block independently — it does not affect the
# main rejection loop. Set INSPECT = False to skip entirely.
# -------------------------------------------------------------------------
INSPECT       = True
AMICA_PROC_PATH = wd / "derivatives" / "05_gedai_gedaiSpec_amica_proc"
INSPECT_SUB   = "sub-01"
INSPECT_SES   = "ses-01"
INSPECT_LABEL = "IC_left"  # "IC_left", "IC_right", "RS_left", "RS_right"

if INSPECT:
    inspect_amica_file = sorted(AMICA_PROC_PATH.rglob(f"*{INSPECT_LABEL}*_amica.amica.npz"))
    inspect_amica_file = [f for f in inspect_amica_file
                          if INSPECT_SUB in f.parts and INSPECT_SES in f.parts]

    if len(inspect_amica_file) == 0:
        print(f"No AMICA file found for {INSPECT_SUB} / {INSPECT_SES} / {INSPECT_LABEL}")
    else:
        inspect_epo_name = inspect_amica_file[0].name.replace("_amica.amica.npz", "-epo.fif")
        inspect_epo_path = EPOCHS_PATH / INSPECT_SUB / INSPECT_SES / "eeg" / inspect_epo_name

        if not inspect_epo_path.exists():
            print(f"Epoch file not found : {inspect_epo_path.name}")
        else:
            e_inspect   = mne.read_epochs(inspect_epo_path, preload=True)
            amica_insp  = AmicaICA.load(str(inspect_amica_file[0]))
            ica_inspect = amica_insp.to_mne_ica()

            ic_labels_insp = label_components(e_inspect, ica_inspect, method="iclabel")
            labs_insp  = ic_labels_insp["labels"]
            probs_insp = ic_labels_insp["y_pred_proba"]

            # -------------------------------------------------------------------------
            # PLOT COMPONENTS WITH ICLABEL PROBABILITIES
            # plot_components returns figure(s) — iterate axes and add label + prob
            # as a subtitle under each topography.
            # -------------------------------------------------------------------------
            figs = ica_inspect.plot_components(
                inst=e_inspect,
                title=f"{INSPECT_SUB} / {INSPECT_SES} / {INSPECT_LABEL}",
                show=False
            )

            # plot_components may return a single figure or a list
            if not isinstance(figs, list):
                figs = [figs]

            for fig in figs:
                for ax in fig.axes:
                    # Extract component index from axis title (e.g. "ICA000")
                    ax_title = ax.get_title()
                    if not ax_title.startswith("ICA"):
                        continue
                    try:
                        comp_idx = int(ax_title.replace("ICA", ""))
                        lab  = labs_insp[comp_idx]
                        prob = probs_insp[comp_idx].max()
                        ax.set_title(f"IC{comp_idx:02d}\n{lab}\n{prob:.2f}", fontsize=7)
                    except (ValueError, IndexError):
                        continue
                fig.tight_layout()
                fig.show()

            # -------------------------------------------------------------------------
            # PLOT SOURCES — click on time course to open detailed component view
            # -------------------------------------------------------------------------
            ica_inspect.plot_sources(e_inspect, block=True)

# 6. AMICA Rejection
Label components with ICLabel and reject artifact components. 
In this study, we used a conservative approach retaining most of the signal by using a threshold of 85% for "eye blink", "eye movement" or "muscle artifact".

In [ ]:
# -------------------------------------------------------------------------
# AMICA COMPONENT REJECTION
# Loads fitted AMICA objects from 05_gedai_gedaiSpec_amica_proc/ and
# corresponding epochs from 04_gedai_gedaiSpec_epochs/.
# -------------------------------------------------------------------------
wd = Path.cwd()
EPOCHS_PATH = wd / "derivatives" / "04_gedai_gedaiSpec_epochs"
AMICA_PROC_PATH = wd / "derivatives" / "05_gedai_gedaiSpec_amica_proc"

# -------------------------------------------------------------------------
# CONFIGURATION
# AMICA_EPOCH_TYPES: "IC", "RS", or None (all epoch types)
# -------------------------------------------------------------------------
AMICA_EPOCH_TYPES = None    # "IC", "RS", or None

# -------------------------------------------------------------------------
# DISCOVER AMICA FILES BASED ON EPOCH TYPE SELECTION
# -------------------------------------------------------------------------
if AMICA_EPOCH_TYPES == "IC":
    amica_pattern = "*IC_*_amica.amica.npz"
elif AMICA_EPOCH_TYPES == "RS":
    amica_pattern = "*RS_*_amica.amica.npz"
elif AMICA_EPOCH_TYPES is None:
    amica_pattern = "*_amica.amica.npz"
else:
    raise ValueError(f"Unknown AMICA_EPOCH_TYPES: '{AMICA_EPOCH_TYPES}'. Choose 'IC', 'RS', or None.")

amica_files = sorted(AMICA_PROC_PATH.rglob(amica_pattern))
print(f"Found {len(amica_files)} fitted AMICA objects (AMICA_EPOCH_TYPES='{AMICA_EPOCH_TYPES}')")

## 6.1. Soft rejection
Only reject IC with > 85 % artifact probability (eye, muscle)

In [ ]:
# -------------------------------------------------------------------------
# AMICA COMPONENT REJECTION
# Loads fitted AMICA objects from 05_gedai_gedaiSpec_amica_proc/ and
# corresponding epochs from 04_gedai_gedaiSpec_epochs/. Applies ICLabel
# classification and excludes components based on the selected rejection
# mode. Cleaned epochs are saved to 06a_gedai_gedaiSpec_amica_<tag>/
# -------------------------------------------------------------------------


# -------------------------------------------------------------------------
# CONFIGURATION
# Two rejection modes:
#   "category" — reject components whose top label matches a category
#                below AND whose confidence exceeds that category's
#                threshold (e.g. eye blink, muscle artifact)
#   "brain"    — keep only components whose top label is "brain" AND
#                confidence exceeds BRAIN_THRESHOLD; reject everything else
# -------------------------------------------------------------------------
REJECTION_MODE = "category"  # "category" or "brain"
REJ_TAG        = "rej85" # e.g. "brain50" or "rej85"

AMICA_REJ_PATH = wd / "derivatives" / f"06a_gedai_gedaiSpec_amica_{REJ_TAG}"


# --- Mode: "category" ---
EXCLUDE_CONFIG = {
    "eye blink"       : 0.85,
    "muscle artifact" : 0.85,
    "brain"           : None,
    "heart beat"      : None,
    "line noise"      : None,
    "channel noise"   : None,
    "other"           : None,
}

# --- Mode: "brain" ---
BRAIN_THRESHOLD = 0.50

print(f"Rejection mode : {REJECTION_MODE}")
print(f"Rejection tag  : {REJ_TAG}")

for amica_path in amica_files:
    ses_name = amica_path.parent.parent.name
    sub_name = amica_path.parent.parent.parent.name

    for candidate in ["IC_left", "IC_right", "RS_left", "RS_right"]:
        if candidate in amica_path.name:
            label = candidate
            break
    else:
        print(f"  Could not determine label from filename, skipping : {amica_path.name}")
        continue

    print(f"\nRejecting components : {sub_name} / {ses_name} / {label}")

    try:
        epo_name = amica_path.name.replace("_amica.amica.npz", "-epo.fif")
        epo_path = EPOCHS_PATH / sub_name / ses_name / "eeg" / epo_name

        if not epo_path.exists():
            print(f"  Epoch file not found, skipping : {epo_path.name}")
            continue

        e = mne.read_epochs(epo_path, preload=True)

        amica   = AmicaICA.load(str(amica_path))
        ica_mne = amica.to_mne_ica()

        ic_labels = label_components(e, ica_mne, method="iclabel")
        labs  = ic_labels["labels"]
        probs = ic_labels["y_pred_proba"]  # top-label confidence per component

        # ---------------------------------------------------------------------
        # IDENTIFY COMPONENTS TO EXCLUDE, BASED ON SELECTED MODE
        # ---------------------------------------------------------------------
        if REJECTION_MODE == "category":
            exclude_idx = []
            for idx, (lab, p) in enumerate(zip(labs, probs)):
                threshold = EXCLUDE_CONFIG.get(lab)
                if threshold is not None and p > threshold:
                    exclude_idx.append(idx)

        elif REJECTION_MODE == "brain":
            exclude_idx = [
                idx for idx, (lab, p) in enumerate(zip(labs, probs))
                if not (lab == "brain" and p > BRAIN_THRESHOLD)
            ]

        else:
            raise ValueError(f"Unknown REJECTION_MODE: '{REJECTION_MODE}'. Choose 'category' or 'brain'.")

        print(f"  Excluding {len(exclude_idx)} / {len(labs)} components:")
        for idx in exclude_idx:
            print(f"    IC{idx:02d} : {labs[idx]} ({probs[idx]:.2f})")

        ep_rej          = e.copy()
        ica_mne.exclude = exclude_idx
        ica_mne.apply(ep_rej)

        
        output_dir = AMICA_REJ_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = epo_name.replace("-epo.fif", f"_desc-amicaRej_{REJ_TAG}-epo.fif")
        output_path = output_dir / output_name

        ep_rej.save(output_path, overwrite=True)
        print(f"  Saved : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} / {label} : {ex}")

    finally:
        for var in ["e", "amica", "ica_mne", "ep_rej"]:
            if var in locals():
                del locals()[var]
        gc.collect()

## 6.2. Hard rejection
Only keep ICs with brain probability > 50 %

In [ ]:
# -------------------------------------------------------------------------
# AMICA COMPONENT REJECTION
# Loads fitted AMICA objects from 05_gedai_gedaiSpec_amica_proc/ and
# corresponding epochs from 04_gedai_gedaiSpec_epochs/. Applies ICLabel
# classification and excludes components based on the selected rejection
# mode. Cleaned epochs are saved to 06b_gedai_gedaiSpec_amica_<tag>/
# -------------------------------------------------------------------------


# -------------------------------------------------------------------------
# CONFIGURATION
# Two rejection modes:
#   "category" — reject components whose top label matches a category
#                below AND whose confidence exceeds that category's
#                threshold (e.g. eye blink, muscle artifact)
#   "brain"    — keep only components whose top label is "brain" AND
#                confidence exceeds BRAIN_THRESHOLD; reject everything else
# -------------------------------------------------------------------------
REJECTION_MODE = "brain"  # "category" or "brain"
REJ_TAG        = "brain50" # e.g. "brain50" or "rej85"

AMICA_REJ_PATH = wd / "derivatives" / f"06b_gedai_gedaiSpec_amica_{REJ_TAG}"


# --- Mode: "category" ---
EXCLUDE_CONFIG = {
    "eye blink"       : 0.85,
    "muscle artifact" : 0.85,
    "brain"           : None,
    "heart beat"      : None,
    "line noise"      : None,
    "channel noise"   : None,
    "other"           : None,
}

# --- Mode: "brain" ---
BRAIN_THRESHOLD = 0.50

print(f"Rejection mode : {REJECTION_MODE}")
print(f"Rejection tag  : {REJ_TAG}")

for amica_path in amica_files:
    ses_name = amica_path.parent.parent.name
    sub_name = amica_path.parent.parent.parent.name

    for candidate in ["IC_left", "IC_right", "RS_left", "RS_right"]:
        if candidate in amica_path.name:
            label = candidate
            break
    else:
        print(f"  Could not determine label from filename, skipping : {amica_path.name}")
        continue

    print(f"\nRejecting components : {sub_name} / {ses_name} / {label}")

    try:
        epo_name = amica_path.name.replace("_amica.amica.npz", "-epo.fif")
        epo_path = EPOCHS_PATH / sub_name / ses_name / "eeg" / epo_name

        if not epo_path.exists():
            print(f"  Epoch file not found, skipping : {epo_path.name}")
            continue

        e = mne.read_epochs(epo_path, preload=True)

        amica   = AmicaICA.load(str(amica_path))
        ica_mne = amica.to_mne_ica()

        ic_labels = label_components(e, ica_mne, method="iclabel")
        labs  = ic_labels["labels"]
        probs = ic_labels["y_pred_proba"]  # top-label confidence per component

        # ---------------------------------------------------------------------
        # IDENTIFY COMPONENTS TO EXCLUDE, BASED ON SELECTED MODE
        # ---------------------------------------------------------------------
        if REJECTION_MODE == "category":
            exclude_idx = []
            for idx, (lab, p) in enumerate(zip(labs, probs)):
                threshold = EXCLUDE_CONFIG.get(lab)
                if threshold is not None and p > threshold:
                    exclude_idx.append(idx)

        elif REJECTION_MODE == "brain":
            exclude_idx = [
                idx for idx, (lab, p) in enumerate(zip(labs, probs))
                if not (lab == "brain" and p > BRAIN_THRESHOLD)
            ]

        else:
            raise ValueError(f"Unknown REJECTION_MODE: '{REJECTION_MODE}'. Choose 'category' or 'brain'.")

        print(f"  Excluding {len(exclude_idx)} / {len(labs)} components:")
        for idx in exclude_idx:
            print(f"    IC{idx:02d} : {labs[idx]} ({probs[idx]:.2f})")

        ep_rej          = e.copy()
        ica_mne.exclude = exclude_idx
        ica_mne.apply(ep_rej)

        
        output_dir = AMICA_REJ_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = epo_name.replace("-epo.fif", f"_desc-amicaRej_{REJ_TAG}-epo.fif")
        output_path = output_dir / output_name

        ep_rej.save(output_path, overwrite=True)
        print(f"  Saved : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} / {label} : {ex}")

    finally:
        for var in ["e", "amica", "ica_mne", "ep_rej"]:
            if var in locals():
                del locals()[var]
        gc.collect()

# Summary

This notebook covers the preprocessing steps applied to the concatenated raw EEG data, preparing it for ICA-based artifact removal.

1. [Setting channel types and montage](#1-channel-types-montage) —
Non-EEG channels were assigned their correct types: `HEOG` and `VEOG` as EOG, `NeckEMG` as EMG, and `x_dir`, `y_dir`, `z_dir` as miscellaneous (acceleration sensors). The standard EasyCap M1 montage was applied to assign electrode locations.

2. [Filtering](#2-filter) —
A high-pass filter of 1 Hz was applied (together with the 131 Hz online low-pass filtering). Note that for future studies a 1 Hz high-pass only is recommended, leaving the low-pass cutoff (40 Hz) to be applied after full processing (after AMICA).

3. [Restoring the reference channel FCz](#3-restoring-reference-channel-and-average-reference) —
FCz was used as the online reference during recording and is therefore absent from the raw data. It was added back as a zero channel and the data was re-referenced to the average of all EEG channels.

4. [Epoching](#4-epoching) —
Epochs were created for direction cue (RS) and initial ground contact (IC) events, separately for left and right sidecuts, and ATR was attached as metadata. Processed files were saved to `derivatives/03_epochs/sub-XX/ses-XX/eeg/`.

5. [GEDAI](#5-gedai) —
The GEDAI pipeline was used for artifact rejection, applied at the epoch level. First the broadband ("normal") version was run, followed by the spectral version on top of its output, using a wavelet decomposition to allow more targeted, frequency-band-specific cleaning.

6. [Adaptive mixture independent component analysis (AMICA)](#6-adaptive-mixture-independent-component-analysis) —
AMICA was fitted per subject and session on the GEDAI-cleaned epochs to decompose the signal into independent components. Components were classified using ICLabel and rejected according to the configured criterion (category-based thresholds or brain-probability-only).

# Check: Event counts

In [7]:
def count_annotations(path):
    path = Path(path)

    fif_files = sorted(path.rglob("*.fif")) if path.is_dir() else [path]

    for fif_path in fif_files:
        ses_name = fif_path.parent.parent.name
        sub_name = fif_path.parent.parent.parent.name

        raw = mne.io.read_raw_fif(fif_path, preload=False, verbose=False)

        ann_counts = defaultdict(int)
        for ann in raw.annotations:
            ann_counts[ann["description"]] += 1

        print(f"\n  {sub_name} / {ses_name}")
        for desc, count in sorted(ann_counts.items()):
            print(f"    {desc:<20} : {count}")

In [6]:
wd = Path.cwd()
PYPREP_PATH = wd / "derivatives" / "04_pyprep"
ASR_PATH    = wd / "derivatives" / "05_asr"
CONCAT_RAW_PATH = wd / "derivatives" / "02_concat-raw"
FILT_REF_PATH = wd / "derivatives" / "03_filt-ref"

In [10]:
count_annotations(ASR_PATH)


  sub-01 / ses-01
    BAD boundary         : 3
    EDGE boundary        : 3
    IC_left              : 58
    IC_right             : 65
    RS_left              : 58
    RS_right             : 65

  sub-02 / ses-01
    BAD boundary         : 2
    EDGE boundary        : 2
    IC_left              : 42
    IC_right             : 51
    RS_left              : 42
    RS_right             : 51


# Check data rank:

In [9]:
wd = Path.cwd()
PYPREP_PATH = wd / "derivatives" / "04_pyprep"
ASR_PATH    = wd / "derivatives" / "05_asr"
CONCAT_RAW_PATH = wd / "derivatives" / "02_concat-raw"
FILT_REF_PATH = wd / "derivatives" / "03_filt-ref"

In [10]:
# -------------------------------------------------------------------------
# FUNCTION: check_rank
# Loads .fif files from a given path and prints the data rank per file.
# Accepts a directory (searches recursively) or a single .fif file.
#
# Parameters
# ----------
# path    : Path — directory or single .fif file
# picks   : str  — channel type to compute rank for (default: "eeg")
# -------------------------------------------------------------------------
def check_rank(path, picks="eeg"):
    path = Path(path)

    fif_files = sorted(path.rglob("*.fif")) if path.is_dir() else [path]

    for fif_path in fif_files:
        ses_name = fif_path.parent.parent.name
        sub_name = fif_path.parent.parent.parent.name

        try:
            # Read epochs or raw depending on file type
            if "-epo.fif" in fif_path.name:
                data = mne.read_epochs(fif_path, preload=True, verbose=False)
            else:
                data = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)

            rank = mne.compute_rank(data, tol=1e-6, tol_kind="relative")
            print(f"  {sub_name} / {ses_name} / {fif_path.name}")
            print(f"    {picks} rank : {rank.get(picks, 'n/a')}")

        except Exception as e:
            print(f"  Error : {sub_name} / {ses_name} : {e}")

In [11]:
check_rank(PYPREP_PATH)

  sub-01 / ses-01 / sub-01_ses-01_task-sidecut_desc-pyprep_eeg.fif
    eeg rank : 56
  sub-02 / ses-01 / sub-02_ses-01_task-sidecut_desc-pyprep_eeg.fif
    eeg rank : 61


In [19]:
check_rank(ASR_PATH)

  sub-01 / ses-01 / sub-01_ses-01_task-sidecut_desc-asr_eeg.fif
    eeg rank : 56
  sub-01 / ses-01 / sub-01_ses-01_task-sidecut_desc-rasr_eeg.fif
    eeg rank : 22
  sub-02 / ses-01 / sub-02_ses-01_task-sidecut_desc-asr_eeg.fif
    eeg rank : 61
  sub-02 / ses-01 / sub-02_ses-01_task-sidecut_desc-rasr_eeg.fif
    eeg rank : 22
